## Clasificacion de desinformacion usando LLMs con el dataset LIAR

In [6]:
# Clasificación de desinformación usando LLMs and the LIAR dataset

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import pandas as pd
import torch
import os
import requests
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
download_url = f"https://drive.google.com/uc?id=1rFZufYDqLOg8ZrkeHlE_WR7WVwklTipP"

cols = ["ID", "Label", "Statement", "Subject", "Speaker", "Job", "State", "Party", "Barely True", "False", "Half True", "Mostly True", "Pants on Fire"]
data = pd.read_csv(download_url, sep="\t", header=None, names=cols)
data = data[["Statement", "Label"]].dropna().reset_index(drop=True)

In [4]:
data.head()

,Statement,Label
0,immigration,Building a wall on the U.S.-Mexico border will...
1,jobs,Wisconsin is on pace to double the number of l...
2,"military,veterans,voting-record",Says John McCain has done nothing to help the ...
3,"medicare,message-machine-2012,campaign-adverti...",Suzanne Bonamici supports a plan that will cut...
4,"campaign-finance,legal-issues,campaign-adverti...",When asked by a reporter whether hes at the ce...


In [9]:
true_labels = ["true", "mostly-true"]
data["Binary_Label"] = data["Label"].apply(lambda x: "true" if x in true_labels else "false")
data.head()

,Statement,Label,Binary_Label
0,immigration,Building a wall on the U.S.-Mexico border will...,false
1,jobs,Wisconsin is on pace to double the number of l...,false
2,"military,veterans,voting-record",Says John McCain has done nothing to help the ...,false
3,"medicare,message-machine-2012,campaign-adverti...",Suzanne Bonamici supports a plan that will cut...,false
4,"campaign-finance,legal-issues,campaign-adverti...",When asked by a reporter whether hes at the ce...,false


In [10]:
model_name = "google/gemma-2b-it"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    pipe = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)
except:
    print("No se pudo cargar Gemma. Usando modelo de respaldo: facebook/bart-large-mnli")
    pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


No se pudo cargar Gemma. Usando modelo de respaldo: facebook/bart-large-mnli


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
etiquetas_binarias = ["true", "false"]
resultados = []
print(data)

for i, fila in data.iterrows():

  contenido = fila["Statement"]
  prompt = f"Given the following content, determine whether it is misinformative or not: '{contenido}'"
  resultado = pipe(prompt, etiquetas_binarias)
  prediccion = resultado["labels"][0]
  resultados.append({
    "Declaración": contenido,
    "Prompt usado": prompt,
    "Etiqueta real": fila["Binary_Label"],
    "Predicción": prediccion
  })

                                              Statement  \
0                                           immigration   
1                                                  jobs   
2                       military,veterans,voting-record   
3     medicare,message-machine-2012,campaign-adverti...   
4     campaign-finance,legal-issues,campaign-adverti...   
...                                                 ...   
1262                                          education   
1263                civil-rights,crime,criminal-justice   
1264     bipartisanship,congress,foreign-policy,history   
1265                  environment,government-efficiency   
1266                  state-budget,state-finances,taxes   

                                                  Label Binary_Label  
0     Building a wall on the U.S.-Mexico border will...        false  
1     Wisconsin is on pace to double the number of l...        false  
2     Says John McCain has done nothing to help the ...        false  
3     S

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [12]:
preds = [r["Predicción"] for r in resultados]
y_true = [r["Etiqueta real"] for r in resultados]

accuracy = accuracy_score(y_true, preds)
precision = precision_score(y_true, preds)
recall = recall_score(y_true, preds)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
resultados_df = pd.DataFrame(resultados)
pd.set_option("display.max_colwidth", None)
print("\nResultados del modelo (etiquetas binarias):\n")
display(resultados_df)

print("\n📊 Métricas de evaluación:")
print(f"Precisión (accuracy): {accuracy:.2f}")
print(f"Precisión positiva (precision): {precision:.2f}")
print(f"Sensibilidad (recall): {recall:.2f}")


Resultados del modelo (etiquetas binarias):



,Declaración,Prompt usado,Etiqueta real,Predicción
0,immigration,"Given the following content, determine whether it is misinformative or not: 'immigration'",false,false
1,jobs,"Given the following content, determine whether it is misinformative or not: 'jobs'",false,false
2,"military,veterans,voting-record","Given the following content, determine whether it is misinformative or not: 'military,veterans,voting-record'",false,false
3,"medicare,message-machine-2012,campaign-advertising","Given the following content, determine whether it is misinformative or not: 'medicare,message-machine-2012,campaign-advertising'",false,false
4,"campaign-finance,legal-issues,campaign-advertising","Given the following content, determine whether it is misinformative or not: 'campaign-finance,legal-issues,campaign-advertising'",false,false
...,...,...,...,...
1262,education,"Given the following content, determine whether it is misinformative or not: 'education'",false,false
1263,"civil-rights,crime,criminal-justice","Given the following content, determine whether it is misinformative or not: 'civil-rights,crime,criminal-justice'",false,false
1264,"bipartisanship,congress,foreign-policy,history","Given the following content, determine whether it is misinformative or not: 'bipartisanship,congress,foreign-policy,history'",false,false
1265,"environment,government-efficiency","Given the following content, determine whether it is misinformative or not: 'environment,government-efficiency'",false,false



📊 Métricas de evaluación:
Precisión (accuracy): 1.00
Precisión positiva (precision): 0.00
Sensibilidad (recall): 0.00


In [15]:
print("\n📋 Informe de clasificación:")
reporte = classification_report(y_true, preds, target_names=["falso", "verdadero"])
print(reporte)


📋 Informe de clasificación:


ValueError: Number of classes, 1, does not match size of target_names, 2. Try specifying the labels parameter

In [ ]:
sns.set(style="whitegrid")
sns.countplot(x="Predicción", data=resultados_df, palette="Set2")
plt.title("Distribución de predicciones del modelo")
plt.xlabel("Etiqueta predicha")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
errores_df = resultados_df[resultados_df["Etiqueta real"] != resultados_df["Predicción"]]
print("\n❌ Ejemplos mal clasificados:")
display(errores_df.head(5))

In [ ]:
# 13. (Opcional) Guardar como CSV
# resultados_df.to_csv("resultados_llm_gemma_liar_completo.csv", index=False)
# errores_df.to_csv("errores_llm_gemma_liar.csv", index=False)